In [8]:
import os
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras import Input
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score,classification_report,confusion_matrix

In [10]:
X_train=pd.read_csv("data/sampled/X_train_sample.csv")
X_test=pd.read_csv("data/sampled/X_test.csv")
Y_train=pd.read_csv("data/sampled/Y_train_sample.csv").squeeze()
Y_test=pd.read_csv("data/sampled/Y_test.csv").squeeze()



In [14]:
le = LabelEncoder()     #encode
Y_train = le.fit_transform(Y_train)
Y_test = le.transform(Y_test)
joblib.dump(le,"models/label_encoder.pk1")

['models/label_encoder.pk1']

In [16]:
print(X_train.shape)  #verify dataset
print(X_test.shape)
print(Y_train.shape)
print(Y_test.shape)

(300000, 78)
(504160, 78)
(300000,)
(504160,)


In [18]:
X_train=X_train.to_numpy()  #convert to numpy
X_test=X_test.to_numpy()

In [54]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
joblib.dump(scaler, "models/lstm_scaler.pkl")

ValueError: Found array with dim 3. StandardScaler expected <= 2.

In [22]:
X_train=X_train.reshape(    #reshape for LSTM
    X_train.shape[0],
    1,
    X_train.shape[1]
)
X_test=X_test.reshape(
    X_test.shape[0],
    1,
    X_test.shape[1]
)

In [26]:
print(X_train.shape)  #verifying reshape
print(X_test.shape)

(300000, 1, 78)
(504160, 1, 78)


In [30]:
model=Sequential([                            #build model
    Input(shape=(1,X_train.shape[2])),
    LSTM(128),
    Dropout(0.30),
    Dense(64,activation="relu"),
    Dropout(0.30),
    Dense(32,activation="relu"),
    Dense(len(np.unique(Y_train)),activation="softmax")
])

In [32]:
model.compile(                         #compile model
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [34]:
model.summary()  #model summary

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ lstm_1 (LSTM)                        │ (None, 128)                 │         105,984 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_2 (Dropout)                  │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ (None, 64)                  │           8,256 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_3 (Dropout)                  │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_4 (Dense)                      │ (None, 32)                  │           2,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_5 (Dense)                      │ (None, 15)                  │             495 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 116,815 (456.31 KB)

 Trainable params: 116,815 (456.31 KB)

 Non-trainable params: 0 (0.00 B)

In [36]:
early_stop=EarlyStopping(           #early stopping
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

In [38]:
checkpoint=ModelCheckpoint(
    "models/lstm.keras",
    monitor="val_accuracy",
    save_best_only=True
)

In [42]:
start_train=time.time()
history=model.fit(
    X_train,
    Y_train,
    epochs=20,
    batch_size=512,
    validation_split=0.20,
    callbacks=[early_stop,checkpoint],
    verbose=1
)
end_train=time.time()
training_time=end_train-start_train
print(f"Training Time: {training_time:.2f} seconds")

Epoch 1/20
469/469 ━━━━━━━━━━━━━━━━━━━━ 11s 16ms/step - accuracy: 0.9270 - loss: 0.3296 - val_accuracy: 0.9646 - val_loss: 0.1032
Epoch 2/20
469/469 ━━━━━━━━━━━━━━━━━━━━ 6s 12ms/step - accuracy: 0.9646 - loss: 0.1007 - val_accuracy: 0.9717 - val_loss: 0.0751
Epoch 3/20
469/469 ━━━━━━━━━━━━━━━━━━━━ 6s 13ms/step - accuracy: 0.9700 - loss: 0.0791 - val_accuracy: 0.9752 - val_loss: 0.0627
Epoch 4/20
469/469 ━━━━━━━━━━━━━━━━━━━━ 6s 13ms/step - accuracy: 0.9729 - loss: 0.0692 - val_accuracy: 0.9791 - val_loss: 0.0573
Epoch 5/20
469/469 ━━━━━━━━━━━━━━━━━━━━ 6s 12ms/step - accuracy: 0.9747 - loss: 0.0630 - val_accuracy: 0.9786 - val_loss: 0.0551
Epoch 6/20
469/469 ━━━━━━━━━━━━━━━━━━━━ 6s 12ms/step - accuracy: 0.9759 - loss: 0.0587 - val_accuracy: 0.9802 - val_loss: 0.0492
Epoch 7/20
469/469 ━━━━━━━━━━━━━━━━━━━━ 6s 13ms/step - accuracy: 0.9772 - loss: 0.0553 - val_accuracy: 0.9811 - val_loss: 0.0468
Epoch 8/20
469/469 ━━━━━━━━━━━━━━━━━━━━ 6s 13ms/step - accuracy: 0.9784 - loss: 0.0527 - val_acc

In [44]:
model=load_model("models/lstm.keras") #load best model

In [48]:
start_predict=time.time()
prob=model.predict(X_test)                   #prediction
Y_pred=np.argmax(prob,axis=1)
end_predict=time.time()
prediction_time=end_predict-start_predict
print(f"Prediction Time: {prediction_time:.2f} seconds")

15755/15755 ━━━━━━━━━━━━━━━━━━━━ 23s 1ms/step
Prediction Time: 34.24 seconds


In [50]:
accuracy = accuracy_score(Y_test, Y_pred)  #accuracy
precision = precision_score(
    Y_test,
    Y_pred,
    average="weighted",
    zero_division=0
)
recall = recall_score(
    Y_test,
    Y_pred,
    average="weighted",
    zero_division=0
)
f1 = f1_score(
    Y_test,
    Y_pred,
    average="weighted",
    zero_division=0
)
print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)

Accuracy : 0.9849155030149159
Precision: 0.9842657784887076
Recall   : 0.9849155030149159
F1 Score : 0.9842831243183172


In [52]:
from sklearn.metrics import balanced_accuracy_score
balanced_acc = balanced_accuracy_score(
    Y_test,
    Y_pred
)
print("Balanced Accuracy :", balanced_acc)

Balanced Accuracy : 0.6142842537822926
